# Lab 5.3 &mdash; An Incident Responder That Knows When to Stop

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- Build the retry cycle: an edge that points backwards, and the budget that ends it
- Route the three ways out of <code>verify</code> &mdash; and see which one is the infinite loop
- Prove a never-healing incident terminates, rather than hoping it does
- Write the escalation handoff: what an on-call human needs at 3am

> **How this lab works.** You write real LangGraph code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a compiled `StateGraph`, a declared reducer, a routing key), so they are deterministic
> and never depend on the model. Cells marked **Run it for real** put your work in front of
> the sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **The system on slide 4.** An alert is triaged against a runbook, a fix is attempted and
> verified, failures cycle back with backoff, and a spent budget hands off to a human.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def _is_todo(exc: BaseException) -> bool:
    """Is this exception really an unfilled blank?

    LangGraph runs your nodes inside tasks, so the NameError from an unfilled BLANK can
    arrive wrapped. Walk the cause chain before calling anything a failure -- telling you
    your answer is wrong when you have not written one yet is the worst thing a lab does.
    """
    seen = set()
    while exc is not None and id(exc) not in seen:
        if isinstance(exc, NameError):
            return True
        seen.add(id(exc))
        exc = exc.__cause__ or exc.__context__
    return False

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except Exception as exc:
        if _is_todo(exc):
            print(f"[TODO] {name}")
            _results.append(None)
            return
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except Exception as exc:
        if not _is_todo(exc):
            raise
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because the "Run it for real" cells make many small calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# Three production incidents and the runbook for each. `heals_on_attempt` makes the fake
# remediation deterministic, so retry behaviour is exact offline -- no sleeping, no luck.
# This case file is Module 5 lab 5.3 only -- 5.1 and 5.2 are different systems.

INCIDENTS = {
    "INC-901": {"symptom": "checkout p99 above 2s",     "service": "checkout",
                "heals_on_attempt": 2},        # a flap: the second try holds
    "INC-902": {"symptom": "payment webhooks 500ing",   "service": "webhook",
                "heals_on_attempt": 1},        # the runbook works first time
    "INC-903": {"symptom": "disk 96% on db-primary",    "service": "db",
                "heals_on_attempt": None},     # no safe fix exists -- a human must decide
}

RUNBOOK = {
    "checkout": "recycle the slowest pod in the checkout deployment",
    "webhook":  "replay the dead-letter queue for the last 15 minutes",
    "db":       "extend the volume -- capacity changes need a human",
}

def apply_fix(ref: str, attempt: int) -> bool:
    """The remediation, standing in for kubectl. Deterministic on purpose."""
    heals = INCIDENTS[ref]["heals_on_attempt"]
    return heals is not None and attempt >= heals

print(f"{len(INCIDENTS)} incidents, {len(RUNBOOK)} runbook entries")

## Concept

A cycle is just an edge pointing at a node that has already run. What makes it safe is not the
edge &mdash; it is the **budget**, and where the budget lives.

A per-node retry count multiplies: three nodes with three attempts each is twenty-seven calls, and
people write per-node counts because that is where the code is. A budget that belongs to the
**run**, decremented by every attempt, is the only thing standing between a bug and a bill.

Then the part worth arguing about: running out of attempts is **not an error**. It is a routing
decision, and the way out is a node.

## Section 1 &mdash; The cycle, and the way out of it

`verify` has three ways out and they all have to exist in the path map. Two of them are yours:
which key the failure-but-attempts-remain case returns, and which node the retry edge points
back at.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END


class IncidentState(TypedDict):
    ref: str
    symptom: str
    runbook_step: str | None
    attempts: int                     # NOT annotated: each attempt sets it, nothing merges
    budget: int                       # a property of the RUN, set once at the top
    healthy: bool
    outcome: str | None
    handoff: dict | None
    log: Annotated[list, add]         # append: what was tried, and what happened each time


def triage(state: IncidentState) -> dict:
    """Match the symptom to the runbook. One lookup; a real one would ask a model."""
    step = RUNBOOK[INCIDENTS[state["ref"]]["service"]]
    return {"runbook_step": step, "log": [f"triaged: {step}"]}


def backoff(attempt: int) -> float:
    """1s, 2s, 4s in production. Scaled down here so the lab does not sleep for a minute."""
    return 0.01 * (2 ** (attempt - 1))


def remediate(state: IncidentState) -> dict:
    """Attempt the runbook step. Charges one attempt against the run's budget."""
    attempt = state["attempts"] + 1
    time.sleep(backoff(attempt))
    worked = apply_fix(state["ref"], attempt)
    return {"attempts": attempt, "healthy": worked,
            "log": [f"attempt {attempt}/{state['budget']}: "
                    f"{'held' if worked else 'did not hold'}"]}


def verify(state: IncidentState) -> dict:
    """Did it hold? Separate from remediate because checking is not fixing."""
    return {"log": [f"verified: {'healthy' if state['healthy'] else 'still failing'}"]}


def next_step(state: IncidentState) -> str:
    """Which way out of verify. Every key returned here must exist in the path map."""
    if state["healthy"]:
        return "resolved"
    if state["attempts"] >= state["budget"]:
        # Out of attempts, and still broken. Returning "retry" here is the infinite loop:
        # the budget test would never be reached again with a different answer.
        return "escalate"
    return "retry"


def resolved(state: IncidentState) -> dict:
    return {"outcome": "resolved", "log": ["closed"]}

## Section 2 &mdash; The handoff

`escalate` is the node the graph reaches when it has run out of safe things to try. A page that
says *"automation failed"* is a page. A page that says what was tried, and what happened each
time, is a **case** &mdash; and the difference is one key of state.

In [ ]:
def escalate(state: IncidentState) -> dict:
    """Assemble the case a human can act on. Every piece of it is already in state."""
    return {"outcome": "escalated",
            "handoff": {
                "incident":       state["ref"],
                "symptom":        state["symptom"],
                "runbook_step":   state["runbook_step"],
                "attempts_spent": state["attempts"],
                # The log is the whole reason it was annotated with a reducer: every node
                # appended to it, so it is the narrative of the run.
                "what_happened":  state["log"],
            },
            "log": ["escalated to the on-call"]}


def build_responder():
    g = StateGraph(IncidentState)
    g.add_node("triage", triage)
    g.add_node("remediate", remediate)
    g.add_node("verify", verify)
    g.add_node("resolved", resolved)
    g.add_node("escalate", escalate)

    g.add_edge(START, "triage")
    g.add_edge("triage", "remediate")
    g.add_edge("remediate", "verify")
    g.add_conditional_edges("verify", next_step, {
        "resolved": "resolved",
        "escalate": "escalate",
        # Back to remediate, not to triage: the runbook step has not changed, so
        # re-triaging would pay for the diagnosis again on every single attempt.
        "retry":    "remediate",
    })
    g.add_edge("resolved", END)
    g.add_edge("escalate", END)
    return g.compile()


def fresh_incident(ref: str, budget: int = 3) -> dict:
    inc = INCIDENTS[ref]
    return {"ref": ref, "symptom": inc["symptom"], "runbook_step": None,
            "attempts": 0, "budget": budget, "healthy": False,
            "outcome": None, "handoff": None, "log": []}

In [ ]:
# --- Self-check: both sections   (a real graph with a real cycle -- still no model)
def _respond(ref: str, budget: int = 3) -> dict:
    return build_responder().invoke(fresh_incident(ref, budget))

check("the responder compiles, cycle and all",
      lambda: build_responder() is not None)
check("the runbook fix that works first time resolves in one attempt",
      lambda: (_respond("INC-902")["outcome"], _respond("INC-902")["attempts"]) == ("resolved", 1))
check("the flap resolves on the SECOND attempt -- the cycle really ran",
      lambda: (_respond("INC-901")["outcome"], _respond("INC-901")["attempts"]) == ("resolved", 2))
check("the incident with no safe fix terminates instead of looping",
      lambda: _respond("INC-903")["outcome"] == "escalated",
      "returning 'retry' when the budget is spent is an infinite loop, not a retry")
check("and it stops at exactly the budget, not one past it",
      lambda: _respond("INC-903")["attempts"] == 3)
check("the budget belongs to the run, so lowering it lowers the bill",
      lambda: _respond("INC-903", budget=1)["attempts"] == 1)
check("the handoff carries what was tried, not just that it failed",
      lambda: len(_respond("INC-903")["handoff"]["what_happened"]) >= 5,
      "the on-call needs the narrative -- that is what the annotated log was for")
check("the handoff names every attempt and its outcome",
      lambda: sum(1 for line in _respond("INC-903")["handoff"]["what_happened"]
                  if line.startswith("attempt ")) == 3)
check("a resolved incident writes no handoff at all",
      lambda: _respond("INC-902")["handoff"] is None)

def _trace():
    for ref in INCIDENTS:
        out = _respond(ref)
        print(f"  {ref}  {out['outcome']:10} after {out['attempts']} attempt(s)")
    print()
    for line in _respond("INC-903")["log"]:
        print("   ", line)
guard(_trace)

## Run it for real &mdash; the page a human receives

The handoff is a dict. A person at 3am wants three sentences. This is the one node in the graph
that genuinely needs a model, which is worth noticing: the routing, the retrying and the budget
were all better off without one.

In [ ]:
PAGE_SYSTEM = ("You are writing an on-call page. Three sentences, no greeting, no bullet "
               "points: what is broken, what automation already tried and what happened, "
               "and the single decision you need the human to make. Never invent a fact "
               "that is not in the handoff.")

def _page_the_human():
    out = build_responder().invoke(fresh_incident("INC-903"))
    handoff = out["handoff"]
    print(json.dumps(handoff, indent=2)[:500], "\n")
    page = ask(f"Handoff:\n{json.dumps(handoff, indent=2)}", system=PAGE_SYSTEM)
    print("--- the page ---")
    print(textwrap.fill(page, 96))

if llm_ready():
    guard(_page_the_human)

In [ ]:
score()

## Your turn

1. Give `triage` a second runbook step to try when the first one fails twice. You will find the
   retry edge now points at the wrong node &mdash; which is the honest reason that blank was a
   decision and not a formality.
2. Make the budget a *token* budget rather than an attempt count, decremented by what each node
   actually spent. Then ask which of the two a finance team would rather you enforced.
3. `escalate` is a node, so it can do more than write a dict: have it decide *who* to page from
   the service in the incident. Then say what happens when that lookup is wrong at 3am.